<a href="https://colab.research.google.com/github/moysessevan/termodinamica-python/blob/main/cod_4_7_transiente_enchimento_tanque_brentq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install CoolProp -q
import CoolProp.CoolProp as CP
from scipy.optimize import brentq

V_tanque = 40.0
T1 = 50.0 + 273.15
V_L1 = 0.02
V_V1 = V_tanque - V_L1

# Estado inicial 1 (equilibrio liquido-vapor a 50 C)
v_L1 = 1.0 / CP.PropsSI('D', 'T', T1, 'Q', 0, 'Water')
v_V1 = 1.0 / CP.PropsSI('D', 'T', T1, 'Q', 1, 'Water')
u_L1 = CP.PropsSI('U', 'T', T1, 'Q', 0, 'Water')
u_V1 = CP.PropsSI('U', 'T', T1, 'Q', 1, 'Water')

m_L1 = V_L1 / v_L1
m_V1 = V_V1 / v_V1
m1 = m_L1 + m_V1
U1 = m_L1 * u_L1 + m_V1 * u_V1
u1 = U1 / m1

# Corrente de alimentacao (P_e = 1.013 bar, x_e = 0.90)
Pe = 1.013e5
xe = 0.90
he = CP.PropsSI('H', 'P', Pe, 'Q', xe, 'Water')

# Balanco transiente em funcao do titulo final x2
def residuo_transiente(x2):
    v2 = 1.0 / CP.PropsSI('D', 'P', Pe, 'Q', x2, 'Water')
    u2 = CP.PropsSI('U', 'P', Pe, 'Q', x2, 'Water')
    m2 = V_tanque / v2
    return m2 * (u2 - he) - m1 * (u1 - he)

x2_sol = brentq(residuo_transiente, 0.001, 0.999)
v2_sol = 1.0 / CP.PropsSI('D', 'P', Pe, 'Q', x2_sol, 'Water')
m2_sol = V_tanque / v2_sol
me_sol = m2_sol - m1
m_L2 = m2_sol * (1.0 - x2_sol)

print(f"Massa inicial (m1):     {m1:.2f} kg (liq: {m_L1:.2f} kg)")
print(f"Titulo final (x2):      {x2_sol:.4f}")
print(f"Massa admitida (me):    {me_sol:.2f} kg")
print(f"Liquido final (m_L2):   {m_L2:.2f} kg")

Massa inicial (m1):     23.08 kg (liq: 19.76 kg)
Titulo final (x2):      0.5153
Massa admitida (me):    23.27 kg
Liquido final (m_L2):   22.47 kg
